# Structured channel–time covariance for sEMG diffusion
## Mathematical validation and NinaPro DB2 feasibility study

**Status:** preliminary proof-of-concept notebook.  
**Primary purpose:** determine whether context-adaptive channel–time covariance is mathematically correct, numerically stable, and empirically supported strongly enough to justify DDPM training.  
**Data boundary:** the 24 confirmatory DB2 participants remain locked. No diffusion network is trained in this notebook.

This notebook deliberately separates two claims:

1. **Implementation validity:** sampled structured noise has the covariance required by the equation.
2. **Empirical suitability:** the covariance model predicts held-out sEMG dependence better than an identity or population-only model.

Passing the first claim does not prove the second. Passing both is a gate for training the proposed diffusion model.

## 1. Research contract

### Research question

> Under a leakage-resistant cross-subject design, does a regularized separable channel–time covariance explain unseen DB2 recordings, and does limited target calibration improve that explanation beyond a fixed source-population covariance?

### Falsifiable expectations

- Known matrix-normal simulations must reproduce their analytic covariance.
- Every covariance factor must be positive definite and trace normalized.
- A structured model should reduce held-out negative log-likelihood relative to identity covariance if correlated structure is useful.
- Context adaptation should reduce held-out negative log-likelihood relative to the population covariance in at least three of four pilot folds to pass the continuation gate.
- A null or negative result is retained. It means the proposed covariance requires revision before DDPM training.

The independent scientific unit remains the participant. Windows are repeated observations nested within participants, movements, and repetitions.

In [ ]:
import csv
import importlib
import json
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

working_directory = Path.cwd().resolve()
override = os.environ.get("EMG_DIFFUSION_ROOT")
repo_candidates = [
    working_directory,
    *working_directory.parents,
    Path("/home/nvidia/Ayorinde_Workspace/EMG_Diffusion"),
    Path.home() / "research" / "DDPM-EMG-Research",
]
if override:
    repo_candidates.insert(0, Path(override).expanduser().resolve())
REPO_ROOT = next(
    (path for path in repo_candidates if (path / "src" / "emg_diffusion").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository. Set EMG_DIFFUSION_ROOT and rerun. "
        f"Current directory: {working_directory}"
    )
sys.path.insert(0, str(REPO_ROOT / "src"))

import emg_diffusion.analysis.covariance as covariance_module
import emg_diffusion.analysis.pilot as pilot_module
import emg_diffusion.data.windows as windows_module
importlib.reload(covariance_module)
importlib.reload(windows_module)
importlib.reload(pilot_module)

from emg_diffusion.analysis.covariance import (
    apply_robust_channel_scaler,
    ar1_covariance,
    covariance_diagnostics,
    estimate_ar1_coefficient,
    estimate_channel_covariance,
    fit_robust_channel_scaler,
    lag_separability_metrics,
    lagged_cross_covariances,
    validate_dense_matrix_normal_sampler,
    validate_production_projection_sampler,
)
from emg_diffusion.analysis.pilot import (
    run_fold_covariance_feasibility,
    summarize_fold_results,
)
from emg_diffusion.data.windows import (
    load_ninapro_windows,
    read_window_manifest,
    record_mask,
    select_central_windows,
)

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "svg.fonttype": "none",
})
print(f"Repository: {REPO_ROOT}")

## 2. Reproducibility inputs

The participant assignment, repetition policy, audit hashes, and calibration allocations were frozen before this analysis. The notebook reads those records rather than generating a convenient split after seeing outcomes.

The balanced covariance subset uses one deterministic central 1000-ms window from every development participant–movement–repetition trial. This gives

$$16\times17\times6=1632$$

windows. Selecting one window per trial prevents small differences in trial duration or overlapping-window count from changing a participant's influence on the feasibility calculation. The later DDPM training pipeline may use all authorized windows.

In [ ]:
SPLIT_PATH = REPO_ROOT / "data/splits/pilot_participants.json"
LOCK_PATH = REPO_ROOT / "data/splits/confirmatory_lock.json"
MANIFEST_PATH = REPO_ROOT / "outputs/data_audit/pilot_window_manifest.csv"
DATA_ROOT = REPO_ROOT / "data/raw/ninapro_db2/extracted"
OUTPUT_ROOT = REPO_ROOT / "outputs/covariance_feasibility"
FIGURE_ROOT = REPO_ROOT / "outputs/figures/covariance_feasibility"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

plan = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
lock = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
assert lock["status"] == "locked"
assert set(plan["development_subjects"]).isdisjoint(lock["confirmatory_subjects"])

print("Development participants:", plan["development_subjects"])
print("Locked confirmatory participants:", lock["confirmatory_subjects"])
for fold in plan["folds"]:
    print(
        f"Fold {fold['fold']}: train={fold['source_train_subjects']}, "
        f"validation={fold['source_validation_subjects']}, "
        f"pseudo-target={fold['pseudo_target_subjects']}"
    )

## 3. Mathematical model and vectorization convention

Let a clean multichannel window be

$$X_0\in\mathbb{R}^{C\times S},$$

where $C=12$ channels and $S=2000$ time samples. We stack the signal **channel by channel**:

$$\operatorname{vec}_{\mathrm{ch}}(X)=[X_{1,:}^{\top},\ldots,X_{C,:}^{\top}]^{\top}.$$

Under this convention, the proposed joint covariance is

$$\Sigma=\Sigma_{\mathrm{ch}}\otimes\Sigma_{\mathrm{time}}.$$

If ordinary column-major vectorization were used instead, the order would be $\Sigma_{\mathrm{time}}\otimes\Sigma_{\mathrm{ch}}$. This is a storage convention, not a scientific disagreement.

With Cholesky factors $\Sigma_{\mathrm{ch}}=L_{\mathrm{ch}}L_{\mathrm{ch}}^\top$ and $\Sigma_{\mathrm{time}}=L_{\mathrm{time}}L_{\mathrm{time}}^\top$, structured noise is sampled as

$$Z=L_{\mathrm{ch}}EL_{\mathrm{time}}^\top,\qquad E_{c,s}\overset{\mathrm{iid}}{\sim}\mathcal N(0,1).$$

The DDPM marginal becomes

$$X_n=\sqrt{\bar\alpha_n}X_0+\sqrt{1-\bar\alpha_n}\,Z.$$

## 4. Numerical proof of the structured sampler

This section asks only whether the implementation obeys the probability model. It uses known synthetic factors so the correct answer is available analytically.

For $M$ sampled matrices, the channel-major empirical covariance is $\widehat\Sigma_M$. We calculate

$$e_{\mathrm{MC}}=\frac{\|\widehat\Sigma_M-\Sigma_{\mathrm{ch}}\otimes\Sigma_{\mathrm{time}}\|_F}{\|\Sigma_{\mathrm{ch}}\otimes\Sigma_{\mathrm{time}}\|_F}.$$

The preregistered reduced-system check uses $C=4$, $S=32$, $M=100{,}000$, and requires $e_{\mathrm{MC}}\leq0.02$. This is an implementation test, not a DB2 result.

In [ ]:
synthetic_channel = 0.35 * np.ones((4, 4)) + 0.65 * np.eye(4)
synthetic_time = ar1_covariance(32, 0.70)
dense_result = validate_dense_matrix_normal_sampler(
    synthetic_channel,
    synthetic_time,
    sample_count=100_000,
    batch_size=2_000,
    seed=17,
)
dense_pass = dense_result.relative_frobenius_error <= 0.02
print(f"Relative Frobenius error: {dense_result.relative_frobenius_error:.6f}")
print(f"Maximum absolute error: {dense_result.maximum_absolute_error:.6f}")
print(f"Empirical mean norm: {dense_result.empirical_mean_norm:.6f}")
print("Reduced-system gate:", "PASS" if dense_pass else "FAIL")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
limit = np.max(np.abs(dense_result.target_covariance))
axes[0].imshow(dense_result.target_covariance, cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[0].set_title("(a) Analytic covariance")
axes[1].imshow(dense_result.empirical_covariance, cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[1].set_title("(b) Empirical covariance")
difference = dense_result.empirical_covariance - dense_result.target_covariance
difference_limit = np.max(np.abs(difference))
image = axes[2].imshow(difference, cmap="RdBu_r", vmin=-difference_limit, vmax=difference_limit)
axes[2].set_title("(c) Empirical − analytic")
fig.colorbar(image, ax=axes[2], fraction=0.046)
for axis in axes:
    axis.set_xlabel("Channel-major vector index")
    axis.set_ylabel("Channel-major vector index")
plt.show()

## 5. Load a balanced, leakage-safe DB2 subset

The next calculation uses real DB2 signals, but only from the 16 development participants. A central window is selected algorithmically from each active trial; no attractive waveform is selected by eye.

At this point, merely loading a locked participant must raise an error. This protects the confirmatory set from accidental use inside a notebook.

In [ ]:
generator_rows = read_window_manifest(MANIFEST_PATH, window_type="generator")
central_rows = select_central_windows(generator_rows)
expected_windows = 16 * 17 * 6
assert len(central_rows) == expected_windows, (len(central_rows), expected_windows)
assert set(int(row["subject"]) for row in central_rows) == set(plan["development_subjects"])

balanced = load_ninapro_windows(
    central_rows,
    DATA_ROOT,
    confirmatory_subjects=lock["confirmatory_subjects"],
    expected_channels=12,
)
windows = balanced.values
records = balanced.records
print("Balanced tensor shape [window, channel, time]:", windows.shape)
print(f"Memory: {windows.nbytes / 1024**2:.1f} MiB")
print("Unique participants:", len({row['subject'] for row in records}))
print("Unique movements:", len({row['movement'] for row in records}))
print("Repetitions:", sorted({row['repetition'] for row in records}))

## 6. Source-only robust scaling

For each fold and channel $c$, source-training samples determine

$$\widetilde X_{i,c,s}=\frac{X_{i,c,s}-m_c}{r_c+\epsilon},$$

where $m_c$ is the median and $r_c=Q_{0.75,c}-Q_{0.25,c}$ is the interquartile range. Validation and pseudo-target test data never determine these values. Per-window min–max scaling is avoided because it would erase amplitude differences that later fidelity tests must examine.

A small residual temporal mean is removed from each channel only for covariance estimation. This keeps DC offsets from being interpreted as physiological channel dependence.

In [ ]:
fold_one = plan["folds"][0]
train_mask = record_mask(records, subjects=fold_one["source_train_subjects"])
validation_test_mask = record_mask(
    records,
    subjects=fold_one["source_validation_subjects"],
    repetitions=plan["repetition_policy"]["target_test_repetitions"],
)
scaler = fit_robust_channel_scaler(windows[train_mask])
scaled_windows = apply_robust_channel_scaler(windows, scaler)
print("Fold 1 source-training windows:", int(np.sum(train_mask)))
print("Fold 1 validation held-out windows:", int(np.sum(validation_test_mask)))
print("Channel medians:", np.array2string(scaler.center, precision=3))
print("Channel IQR scales:", np.array2string(scaler.scale, precision=3))

## 7. Estimate and stabilize channel covariance

For $N$ source windows,

$$S_{\mathrm{ch}}=\frac{1}{NS}\sum_{i=1}^{N}\widetilde X_i\widetilde X_i^\top.$$

We shrink this estimate toward a scaled identity:

$$\widehat\Sigma_{\mathrm{ch}}(\gamma)=(1-\gamma)S_{\mathrm{ch}}+\gamma\frac{\operatorname{tr}(S_{\mathrm{ch}})}{C}I_C.$$

The initial engineering value is $\gamma=0.01$. After eigenvalue flooring, the matrix is trace normalized so

$$\frac{1}{C}\operatorname{tr}(\widehat\Sigma_{\mathrm{ch}})=1.$$

Trace normalization prevents overall variance from being confused with the DDPM noise schedule.

In [ ]:
population_channel = estimate_channel_covariance(
    scaled_windows[train_mask],
    shrinkage=0.01,
    minimum_eigenvalue=1e-6,
)
channel_diagnostics = covariance_diagnostics(population_channel)
print(json.dumps(channel_diagnostics, indent=2))
assert channel_diagnostics["minimum_eigenvalue"] > 0
assert np.isclose(channel_diagnostics["mean_eigenvalue"], 1.0)

fig, ax = plt.subplots(figsize=(6.2, 5.1), constrained_layout=True)
limit = np.max(np.abs(population_channel))
image = ax.imshow(population_channel, cmap="RdBu_r", vmin=-limit, vmax=limit)
ax.set_title("Fold 1 source-population channel covariance")
ax.set_xlabel("sEMG channel")
ax.set_ylabel("sEMG channel")
ax.set_xticks(range(12), range(1, 13))
ax.set_yticks(range(12), range(1, 13))
fig.colorbar(image, ax=ax, label="Trace-normalized covariance")
plt.show()

## 8. Estimate the first temporal model

The simplest stationary temporal model is AR(1):

$$\widehat\rho=\frac{\sum_{i,c,s=2}^{S}\widetilde X_{i,c,s}\widetilde X_{i,c,s-1}}{\sum_{i,c,s=2}^{S}\widetilde X_{i,c,s-1}^{2}},$$

$$[\widehat\Sigma_{\mathrm{time}}]_{j,k}=\widehat\rho^{|j-k|}.$$

For $|\rho|<1$, this Toeplitz correlation matrix is positive definite. AR(1) is an intentionally restrictive baseline. If its lag curve misses repeatable multiscale structure, we will test a richer non-negative spectral covariance rather than pretending AR(1) is sufficient.

In [ ]:
population_rho = estimate_ar1_coefficient(scaled_windows[train_mask])
lags = (0, 1, 2, 5, 10, 20, 50, 100, 200)
lagged = lagged_cross_covariances(scaled_windows[validation_test_mask], lags)
separability = lag_separability_metrics(lagged)
print(f"Population AR(1) coefficient: {population_rho:.6f}")
print(f"Held-out separability residual: {separability['aggregate_relative_residual']:.6f}")

lag_array = np.asarray(lags)
empirical_coefficients = np.asarray([separability['lag_coefficients'][lag] for lag in lags])
relative_residuals = np.asarray([separability['relative_residuals'][lag] for lag in lags])
ar1_prediction = population_rho ** lag_array

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.1), constrained_layout=True)
axes[0].plot(lag_array / 2.0, empirical_coefficients, "o-", label="Held-out empirical scalar")
axes[0].plot(lag_array / 2.0, ar1_prediction, "--", label="Source AR(1) prediction")
axes[0].set(xlabel="Lag (ms)", ylabel="Normalized lag coefficient", title="(a) Temporal dependence")
axes[0].legend()
axes[0].grid(alpha=0.25)
axes[1].plot(lag_array[1:] / 2.0, relative_residuals[1:], "s-", color="#D55E00")
axes[1].set(xlabel="Lag (ms)", ylabel="Relative Frobenius residual", title="(b) Failure of exact separability")
axes[1].grid(alpha=0.25)
plt.show()

## 9. What the separability test means

For lag $\ell$, define

$$\widehat\Gamma(\ell)=\frac{1}{N(S-\ell)}\sum_{i=1}^{N}\sum_{s=1}^{S-\ell}\widetilde x_{i,:,s}\widetilde x_{i,:,s+\ell}^{\top}.$$

Exact separability requires each lagged channel matrix to be a scalar multiple of the lag-zero matrix. The best scalar is

$$\widehat r_\ell=\frac{\langle\widehat\Gamma(\ell),\widehat\Gamma(0)\rangle_F}{\|\widehat\Gamma(0)\|_F^2}.$$

The aggregate residual is

$$e_{\mathrm{sep}}=\left[\frac{\sum_{\ell\in\mathcal L_+}\|\widehat\Gamma(\ell)-\widehat r_\ell\widehat\Gamma(0)\|_F^2}{\sum_{\ell\in\mathcal L_+}\|\widehat\Gamma(\ell)\|_F^2}\right]^{1/2}.$$

A value near zero supports one separable factor. A large value indicates that channel relationships change with lag and may require a sum of Kronecker components or another structured correction. There is no universal biological cutoff; stability across folds and improvement in held-out likelihood are more important than declaring an arbitrary threshold.

## 10. Production-scale sampler check

A dense covariance for a $12\times2000$ signal would have dimension $24{,}000\times24{,}000$, which is unnecessary and computationally wasteful. Instead, choose fixed projection matrices $P_k$ and compare

$$\operatorname{Var}(\langle P_k,Z\rangle_F)=\operatorname{vec}_{\mathrm{ch}}(P_k)^\top(\Sigma_{\mathrm{ch}}\otimes\Sigma_{\mathrm{time}})\operatorname{vec}_{\mathrm{ch}}(P_k)$$

with the empirical variance of projected production-sized samples. This tests the implemented covariance at the real tensor dimension without materializing the full Kronecker matrix.

In [ ]:
projection_result = validate_production_projection_sampler(
    population_channel,
    population_rho,
    time_count=2000,
    projection_count=12,
    sample_count=5_000,
    batch_size=100,
    seed=42,
)
print(f"Projection RMSE of relative variance: {projection_result.root_mean_squared_relative_error:.6f}")
print(f"Maximum absolute relative error: {projection_result.maximum_absolute_relative_error:.6f}")

fig, ax = plt.subplots(figsize=(5.8, 5.0), constrained_layout=True)
lower = min(projection_result.target_variances.min(), projection_result.empirical_variances.min())
upper = max(projection_result.target_variances.max(), projection_result.empirical_variances.max())
ax.plot([lower, upper], [lower, upper], "--", color="#6B7280", label="Exact agreement")
ax.scatter(projection_result.target_variances, projection_result.empirical_variances, color="#0072B2")
for index, (target, empirical) in enumerate(zip(projection_result.target_variances, projection_result.empirical_variances, strict=True), start=1):
    ax.annotate(str(index), (target, empirical), xytext=(3, 3), textcoords="offset points", fontsize=8)
ax.set(xlabel="Analytic projection variance", ylabel="Empirical projection variance", title="Production-scale covariance check")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 11. Limited-data context adaptation

For a target participant $u$, the calibrated channel covariance is

$$\Sigma_{\mathrm{ch}}^{(u)}(\eta)=(1-\eta)\Sigma_{\mathrm{ch,pop}}+\eta\Sigma_{\mathrm{ch,cal}}^{(u)}.$$

The same first-stage weight adapts the AR(1) coefficient:

$$\rho^{(u)}(\eta)=(1-\eta)\rho_{\mathrm{pop}}+\eta\rho_{\mathrm{cal}}^{(u)}.$$

Using one common $\eta$ limits tuning freedom in the first feasibility study. Separate channel and temporal weights can be considered later only if justified.

For a held-out window $X$, covariance models are compared using matrix-normal negative log-likelihood per scalar observation:

$$\ell(X)=\frac{1}{2CS}\left[S\log|\Sigma_{\mathrm{ch}}|+C\log|\Sigma_{\mathrm{time}}|+\operatorname{tr}(\Sigma_{\mathrm{ch}}^{-1}X\Sigma_{\mathrm{time}}^{-1}X^\top)\right].$$

The common Gaussian constant is omitted. **Lower is better.** The grid $\eta\in\{0,0.1,\ldots,1\}$ is selected only with source-validation participants, then applied unchanged to pseudo-target participants.

In [ ]:
fold_results = []
for fold in plan["folds"]:
    print(f"Running covariance feasibility for fold {fold['fold']}...")
    result = run_fold_covariance_feasibility(
        windows,
        records,
        fold,
        plan["calibration_allocations"],
        test_repetitions=plan["repetition_policy"]["target_test_repetitions"],
        eta_grid=np.linspace(0.0, 1.0, 11),
        lags=lags,
        covariance_shrinkage=0.01,
        minimum_eigenvalue=1e-6,
    )
    fold_results.append(result)

fold_summary = summarize_fold_results(fold_results)
for row in fold_summary:
    print(
        f"fold={row['fold']} budget={row['budget_repetitions']} "
        f"eta={row['eta']:.1f} ΔNLL={row['mean_adapted_minus_population_nll']:+.6f} "
        f"improved={row['improved_subject_count']}/{row['subject_count']} "
        f"e_sep={row['separability_residual']:.4f}"
    )

## 12. Interpretation rules

Define

$$\Delta\mathrm{NLL}=\mathrm{NLL}_{\mathrm{adapted}}-\mathrm{NLL}_{\mathrm{population}}.$$

Therefore:

- $\Delta\mathrm{NLL}<0$: calibration improved covariance prediction.
- $\Delta\mathrm{NLL}=0$: calibration added no predictive information.
- $\Delta\mathrm{NLL}>0$: adaptation overfit or shifted covariance in the wrong direction.

Window-level values are averaged within each participant and calibration allocation. We do not treat 1,632 windows as 1,632 independent participants. The continuation decision is based on fold consistency and participant-level direction, not a window-level p-value.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.2), constrained_layout=True)

fold_one_tuning = fold_results[0]["validation_tuning"]
for budget, color in ((1, "#0072B2"), (2, "#E69F00")):
    rows = [row for row in fold_one_tuning if row["budget_repetitions"] == budget]
    axes[0, 0].plot(
        [row["eta"] for row in rows],
        [row["mean_validation_nll"] for row in rows],
        "o-",
        color=color,
        label=f"{budget}-repetition budget",
    )
axes[0, 0].set(xlabel="Adaptation weight η", ylabel="Mean validation NLL per element", title="(a) Fold-1 validation tuning")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.25)

one_row_per_fold = [row for row in fold_summary if row["budget_repetitions"] == 2]
axes[0, 1].bar(
    [f"Fold {row['fold']}" for row in one_row_per_fold],
    [row["separability_residual"] for row in one_row_per_fold],
    color="#009E73",
    edgecolor="#17212B",
)
axes[0, 1].set(ylabel="Held-out relative residual", title="(b) Separability across folds")
axes[0, 1].grid(axis="y", alpha=0.25)

x_positions = []
deltas = []
labels = []
for fold_result in fold_results:
    for subject in fold_result["pseudo_target_subjects"]:
        subject_rows = [
            row for row in fold_result["pseudo_target_evaluation"]
            if row["subject"] == subject and row["budget_repetitions"] == 2
        ]
        x_positions.append(len(x_positions))
        deltas.append(np.mean([row["adapted_minus_population_nll"] for row in subject_rows]))
        labels.append(f"S{subject}")
colors = ["#0072B2" if value < 0 else "#D55E00" for value in deltas]
axes[1, 0].axhline(0.0, color="#17212B", linewidth=1.0)
axes[1, 0].scatter(x_positions, deltas, c=colors, s=42, edgecolor="#17212B", linewidth=0.4)
axes[1, 0].set_xticks(x_positions, labels, rotation=45, ha="right")
axes[1, 0].set(ylabel="Adapted − population NLL", title="(c) Two-repetition pseudo-target effects")
axes[1, 0].grid(axis="y", alpha=0.25)

width = 0.34
fold_x = np.arange(1, 5)
for offset, budget, color in ((-width / 2, 1, "#0072B2"), (width / 2, 2, "#E69F00")):
    rows = [row for row in fold_summary if row["budget_repetitions"] == budget]
    axes[1, 1].bar(fold_x + offset, [row["eta"] for row in rows], width=width, color=color, edgecolor="#17212B", label=f"{budget} rep.")
axes[1, 1].set_xticks(fold_x, [f"Fold {value}" for value in fold_x])
axes[1, 1].set(xlabel="Outer fold", ylabel="Validation-selected η", title="(d) Frozen adaptation weights", ylim=(0, 1.05))
axes[1, 1].legend()
axes[1, 1].grid(axis="y", alpha=0.25)

figure_title = "DB2 structured-covariance feasibility evidence"
figure_description = (
    "Four-panel summary of the validation-only adaptation search, held-out "
    "channel-time separability residuals, participant-level changes in negative "
    "log-likelihood after two calibration repetitions, and the adaptation weights "
    "selected independently within each outer fold. Blue participant markers "
    "indicate improved held-out likelihood; vermillion markers indicate no improvement."
)
fig.suptitle(figure_title, fontsize=14, fontweight="bold")
figure_stem = FIGURE_ROOT / "covariance_feasibility_summary"
fig.savefig(
    figure_stem.with_suffix(".svg"),
    bbox_inches="tight",
    metadata={"Title": figure_title, "Description": figure_description},
)
fig.savefig(figure_stem.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(figure_stem.with_suffix(".png"), bbox_inches="tight", dpi=300)
caption = (
    "Structured-covariance feasibility results on the frozen DB2 development "
    "participants. Adaptation weights are selected using source-validation "
    "participants only. Pseudo-target effects are computed on repetitions excluded "
    "from calibration, and each point in panel (c) summarizes one participant."
)
figure_stem.with_name(figure_stem.name + "_caption.txt").write_text(
    caption + "\n", encoding="utf-8"
)
print(f"Saved publication figure to {figure_stem}.svg/.pdf/.png")
plt.show()

## 13. Archive the evidence

The raw fold-level output is saved before any prose conclusion is written. This ensures that later proposal claims can be traced back to participant assignments, calibration allocations, equations, and numerical results. Generated files remain outside Git because they are reproducible outputs rather than source code.

In [ ]:
results_payload = {
    "status": "pilot_covariance_feasibility",
    "split_seed": plan["split_seed"],
    "development_subjects": plan["development_subjects"],
    "confirmatory_subjects_accessed": [],
    "balanced_window_count": int(windows.shape[0]),
    "dense_sampler_validation": {
        "sample_count": dense_result.sample_count,
        "relative_frobenius_error": dense_result.relative_frobenius_error,
        "maximum_absolute_error": dense_result.maximum_absolute_error,
        "empirical_mean_norm": dense_result.empirical_mean_norm,
        "threshold": 0.02,
        "passed": bool(dense_pass),
    },
    "production_projection_validation": {
        "sample_count": projection_result.sample_count,
        "root_mean_squared_relative_error": projection_result.root_mean_squared_relative_error,
        "maximum_absolute_relative_error": projection_result.maximum_absolute_relative_error,
    },
    "fold_results": fold_results,
    "fold_summary": fold_summary,
}
results_path = OUTPUT_ROOT / "covariance_feasibility_results.json"
results_path.write_text(json.dumps(results_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

summary_path = OUTPUT_ROOT / "covariance_feasibility_fold_summary.csv"
with summary_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(fold_summary[0]))
    writer.writeheader()
    writer.writerows(fold_summary)
print("Saved:", results_path)
print("Saved:", summary_path)

## 14. How this notebook maps into the dissertation proposal

### Research Methodology

The proposal methodology should contain the research question, vectorization convention, covariance estimators, regularization, trace normalization, AR(1) baseline, separability residual, matrix-normal likelihood, validation-only selection of $\eta$, Monte Carlo protocol, leakage boundary, and continuation rule. These statements describe what is done and why.

### Preliminary Results

Only after this notebook has executed successfully should the proposal report the measured Monte Carlo error, channel covariance diagnostics, AR(1) coefficients, separability residuals, selected adaptation weights, participant-level $\Delta\mathrm{NLL}$ values, failures, and uncertainty. Results must be described as pilot feasibility evidence, not proof of superior prosthetic control or completed DDPM performance.

### What this notebook cannot establish

- It does not show that the diffusion denoiser learns better.
- It does not show improved gesture-classification accuracy.
- It does not establish cross-session or amputee generalization.
- It does not prove the covariance mechanism is globally novel.

Those questions require later controlled experiments.